# Validation et interprétabilité

Le modèle `data/model.pkl` est audité sur `data/X_test.csv` et `data/y_test.csv`.
Les contrôles sont les densités de probabilités, la calibration, la courbe ROC et la courbe précision-rappel.
L'interprétation locale repose sur les valeurs de Shapley calculées par Tree SHAP.
Chaque valeur est une contribution en log-odds, avant la fonction logistique.
Aucun fichier n'est écrit.


## Chargement


In [ ]:
import os
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme()

model = joblib.load(os.path.expanduser("data/model.pkl"))
X_train = pd.read_csv(os.path.expanduser("data/X_train.csv"))
X_test = pd.read_csv(os.path.expanduser("data/X_test.csv"))
y_train = pd.read_csv(os.path.expanduser("data/y_train.csv")).values.flatten()
y_test = pd.read_csv(os.path.expanduser("data/y_test.csv")).values.flatten()

y_prob = model.predict_proba(X_test)


## Densités et calibration

Les histogrammes portent sur la probabilité de la classe positive, séparée selon l'achat réel. La moyenne et la médiane de chaque groupe sont tracées.
La calibration reprend 20 tranches sur le même ensemble de test.


In [ ]:
plt.figure(figsize=(16, 10))

sns.histplot(y_prob[y_test == 0, 1], alpha=0.5)
plt.axvline(np.median(y_prob[y_test == 0, 1]), 0, 1, linestyle="--", label="Median Class 0")
plt.axvline(np.mean(y_prob[y_test == 0, 1]), 0, 1, linestyle="-", label="Mean Class 0")

sns.histplot(y_prob[y_test == 1, 1], color="darkorange", alpha=0.4)
plt.axvline(np.median(y_prob[y_test == 1, 1]), 0, 1, linestyle="--", color="darkorange", label="Median Class 1")
plt.axvline(np.mean(y_prob[y_test == 1, 1]), 0, 1, linestyle="-", color="darkorange", label="Mean Class 1")

plt.legend()
plt.xlabel("Predicted probabilites")
plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(1, 0))
plt.xlim(-0.05, 1.05)
plt.title("Density Chart", fontsize=16)
plt.show()


In [ ]:
from sklearn.calibration import calibration_curve

prob_pos = model.predict_proba(X_test)[:, 1]
fraction_of_positives, mean_predicted_value = calibration_curve(y_test, prob_pos, n_bins=20)

plt.figure(figsize=(16, 10))
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated", alpha=0.6)
plt.plot(mean_predicted_value, fraction_of_positives, "s-", label="Model")
plt.ylabel("Fraction of positives")
plt.xlabel("Predicted probabilites")
plt.legend()
plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(1, 0))
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1, 0))
plt.title("Calibration Curve")
plt.show()


## Courbes ROC et PR

La courbe ROC trace le taux de vrais positifs en fonction du taux de faux positifs. Elle mobilise les quatre cases de la matrice de confusion.
La courbe PR trace la précision en fonction du rappel. Elle mobilise les vrais positifs, les faux positifs et les faux négatifs.
La courbe PR est le graphique adapté à un critère de précision et de rappel.


In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test)[:, 1])

plt.figure(figsize=(16, 10))
plt.plot(
    fpr,
    tpr,
    color="darkorange",
    lw=2,
    label="ROC curve (area = {:2.1f}%)".format(auc(fpr, tpr) * 100),
)
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(1, 0))
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1, 0))
plt.title("ROC Curve", fontsize=16)
plt.legend(loc="lower right")


In [ ]:
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import PrecisionRecallDisplay

import matplotlib.pyplot as plt

plt.figure(figsize=(16, 11))
prec, recall, _ = precision_recall_curve(
    y_test, model.predict_proba(X_test)[:, 1], pos_label=1
)
pr_display = PrecisionRecallDisplay.from_estimator(model, X_test, y_test)
plt.title("PR Curve", fontsize=16)
plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(1, 0))
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1, 0))
plt.show()


## Valeurs de Shapley

Les variables jouent le rôle des joueurs. Le gain réparti est la prédiction.
Pour une observation, la probabilité de la classe positive est la sigmoïde de la valeur de base plus la somme des contributions : 1 / (1 + exp(-(φ0 + Σ φj))).
Tree SHAP calcule ces contributions à partir des arbres. Elles sont exprimées en log-odds.
`product_id`, `brand` et `category` sont des codes d'étiquettes : leur axe n'est pas une échelle ordonnée.
Une contribution se lit avec les autres variables.


In [ ]:
import shap

explainer = shap.TreeExplainer(model)
X_shap = X_test.copy()
shap_values = explainer.shap_values(X_shap)


In [ ]:
def plot_shapley_values(index):
    shap_df = pd.DataFrame.from_dict({
        "Variable": X_shap.columns + " (" + X_shap.iloc[0, :].values.astype(str) + ")",
        "Valeur de Shapley": shap_values[index, :],
    })

    # Log-odds : valeur de base + contributions, puis sigmoïde
    prob = explainer.expected_value + shap_df["Valeur de Shapley"].sum()
    prob = 1 / (1 + np.exp(-prob))

    plt.figure(figsize=(13, 10))
    sns.barplot(
        y="Variable",
        x="Valeur de Shapley",
        data=shap_df.sort_values("Valeur de Shapley", ascending=False),
    )
    plt.title("Probabilité : {:2.2f}%".format(prob * 100), fontsize=18)
    plt.yticks(fontsize=13)

plot_shapley_values(8)


In [ ]:
plot_shapley_values(1)


In [ ]:
shap.summary_plot(shap_values, X_shap, plot_size=0.8)


In [ ]:
shap.dependence_plot("product_id", shap_values, X_shap, interaction_index=None)


In [ ]:
shap.dependence_plot("hour", shap_values, X_shap, interaction_index=None)


In [ ]:
shap.dependence_plot("num_views_session", shap_values, X_shap, interaction_index=None)
